# Batch Runner — Parameter Space Sweep

**Run this notebook overnight.** It loops through all agent count × drainage rate
combinations automatically. Each run is saved to its own folder.

## Experimental Design
- **Duration:** 10 years per run (fixed)
- **Agent counts:** 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100 (11 levels)
- **Recession rates:** 0.1 → 1.0 in 10 steps (slow/wet → fast/drought)
- **Total:** 110 runs

### How to use
1. Set paths in Cell 1
2. **Run All Cells** (Kernel → Restart & Run All)
3. Walk away — progress is printed and logged
4. If interrupted, re-run: completed runs are detected and skipped

### Estimated runtime
Each run (10 years × 365 ticks × N agents) takes roughly 2–15 minutes
depending on agent count. Total: ~6–20 hours for 110 runs.

## Cell 1 — Configuration

In [2]:
from pathlib import Path
import time
import datetime
import os
# ========================================
# PATHS — UPDATE THESE
# ========================================
# ========================================
CSV_PATH = r"C:\Users\Wyss User\Desktop\DigitalBeaverWorld\A_MASTER_veg_stream.csv"
SAVE_PATH = r"H:\DigitalBeaverWorld"
RUNS_BASE_DIR = os.path.join(SAVE_PATH, "runs")
DATA_EPSG = 26913
GRID_SPACING_M = None  # inferred from data if None

# ========================================
# PARAMETER SPACE
# ========================================
N_YEARS = 10
TICKS_PER_YEAR = 365

AGENT_COUNTS = [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 25]

# 10 recession rates: 0.1 (slow/wet) → 1.0 (fast/drought)
import numpy as np
RECESSION_RATES = np.round(np.linspace(0.1, 1.0, 10), 2).tolist()

# ========================================
# FIXED AGENT PARAMETERS
# ========================================
VISION_MIN = 3
VISION_MAX = 20
AGENT_SPEED = 3
EXCAVATION_DEPTH_M = 0.2

# ========================================
# HEMISPHERE EROSION
# ========================================
R_LIMIT = 6  # scaling factor of stream width

# ========================================
# BATCH OPTIONS
# ========================================
SKIP_COMPLETED = True   # Skip runs that already have output folders
VERBOSE_MODEL = False   # Suppress per-tick printing (much faster)
PLOT_DURING_RUN = False # No plots during batch (saves time)

total_runs = len(AGENT_COUNTS) * len(RECESSION_RATES)
print(f"Parameter space: {len(AGENT_COUNTS)} agent levels × {len(RECESSION_RATES)} recession rates = {total_runs} runs")
print(f"Agent counts:    {AGENT_COUNTS}")
print(f"Recession rates: {RECESSION_RATES}")
print(f"Years per run:   {N_YEARS}")
print(f"Skip completed:  {SKIP_COMPLETED}")

Parameter space: 11 agent levels × 10 recession rates = 110 runs
Agent counts:    [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 25]
Recession rates: [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
Years per run:   10
Skip completed:  True


## Cell 2 — Imports

In [3]:
import numpy as np
import pandas as pd
import random
import math
import os
import json
import re
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for batch mode
import matplotlib.pyplot as plt
from pathlib import Path

from mesa import Agent, Model
from pyproj import Transformer
from landlab import RasterModelGrid
from landlab.components.flow_accum import FlowAccumulator

print("All imports loaded.")

All imports loaded.


## Cell 3 — Full Model Code

This is a copy of all functions from the simulation notebook (Cells 3–11).
Pasted here so the batch runner is fully self-contained.

In [4]:
# ========================================================================
# NOTE: Paste the contents of Cells 3 through 11 from
# Mesa_canal_water_greedy_stigmery_UPDATED5.ipynb here.
#
# This includes:
#   - infer_step, create_grid_FIXED
#   - recompute_hydro_fields_landlab
#   - WaterFollowingExcavationAgent
#   - BeaverWorldModel
#   - export_world_from_model, save_year_csv
#   - All visualization functions
#   - make_run_dirs, _save_run_config, save_dem_products, etc.
#   - compute_site_volume, compute_2d_erosion, compute_2d_erosion_fast
#   - build_R_grid_from_volume, run_hemisphere_erosion_pipeline
#   - run_yearly_cycles_v2
#
# To do this automatically, run the cell below instead.
# ========================================================================

In [5]:
# ========================================================================
# AUTO-IMPORT: Execute the simulation notebook's code cells directly.
# This avoids copy-paste and stays in sync with your latest version.
# ========================================================================

import json as _json

# Path to your simulation notebook
_SIM_NOTEBOOK = Path(CSV_PATH).parent / "Mesa_canal_water_greedy_stigmery_UPDATED5_erosion.ipynb"

if _SIM_NOTEBOOK.exists():
    with open(_SIM_NOTEBOOK, 'r', encoding='utf-8') as _f:
        _nb = _json.load(_f)

    _code_cells = [c for c in _nb['cells'] if c['cell_type'] == 'code']
    print(f"Found {len(_code_cells)} code cells in {_SIM_NOTEBOOK.name}")

    # Execute cells 2 through 11 (indices 1-10 of code cells)
    # Cell 0 = config (skip, we have our own)
    # Cell 1 = imports (skip, we already imported)
    # Cells 2-10 = utility, hydro, agent, model, export, viz, dirs, hemisphere, run_v2
    # Cell 11 = execute (skip, we have our own loop)

    # We need cells at code-cell indices 2 through 10 (0-indexed)
    for _idx in range(2, 11):  # cells 2,3,4,5,6,7,8,9,10
        if _idx < len(_code_cells):
            _src = ''.join(_code_cells[_idx]['source'])
            # Suppress PLOT_END_OF_YEAR reference by defining it
            exec(_src, globals())
            print(f"  Executed code cell {_idx}")

    print("\nAll model code loaded from simulation notebook.")
else:
    print(f"ERROR: Simulation notebook not found at {_SIM_NOTEBOOK}")
    print("Either:")
    print("  1. Update _SIM_NOTEBOOK path above, or")
    print("  2. Paste Cells 3-11 into the cell above manually")

# Ensure batch-mode globals exist
if 'PLOT_END_OF_YEAR' not in dir():
    PLOT_END_OF_YEAR = False

Found 14 code cells in Mesa_canal_water_greedy_stigmery_UPDATED5_erosion.ipynb
  Executed code cell 2
  Executed code cell 3
  Executed code cell 4
  Executed code cell 5
  Executed code cell 6
  Executed code cell 7
  Executed code cell 8
  Executed code cell 9
  Executed code cell 10

All model code loaded from simulation notebook.


## Cell 4 — Detect Completed Runs

Scans the runs directory and identifies which parameter combinations
have already been completed, so they can be skipped on re-run.

In [6]:
# ========================================================================
# DETECT COMPLETED RUNS
# ========================================================================

def parse_run_folder_name(name: str) -> dict:
    """Extract parameters from run folder name."""
    pattern = re.compile(r'_Y(\d+)_A(\d+)_R(\d+p?\d*)')
    m = pattern.search(name)
    if not m:
        return None
    return {
        'n_years': int(m.group(1)),
        'n_agents': int(m.group(2)),
        'recession_rate': float(m.group(3).replace('p', '.')),
    }


def is_run_complete(run_path: Path, n_years: int) -> bool:
    """Check if a run folder has all expected outputs for n_years."""
    sv_dir = run_path / 'site_volume'
    ero_dir = run_path / '2d_erosion'
    if not sv_dir.exists() or not ero_dir.exists():
        return False
    # Check that the final year's files exist
    sv_files = list(sv_dir.glob(f'*year{n_years:02d}.csv'))
    ero_files = list(ero_dir.glob(f'*year{n_years:02d}.csv'))
    return len(sv_files) > 0 and len(ero_files) > 0


def find_completed_runs(runs_base: str, n_years: int) -> set:
    """Return set of (n_agents, recession_rate) tuples for completed runs."""
    completed = set()
    runs_path = Path(runs_base)
    if not runs_path.exists():
        return completed
    for d in runs_path.iterdir():
        if not d.is_dir():
            continue
        params = parse_run_folder_name(d.name)
        if params is None:
            continue
        if params['n_years'] == n_years and is_run_complete(d, n_years):
            completed.add((params['n_agents'], round(params['recession_rate'], 2)))
    return completed


completed_runs = find_completed_runs(RUNS_BASE_DIR, N_YEARS)
print(f"Found {len(completed_runs)} completed runs out of {total_runs} total.")
remaining = total_runs - len(completed_runs)
print(f"Remaining to run: {remaining}")

if len(completed_runs) > 0:
    print("\nCompleted:")
    for a, r in sorted(completed_runs):
        print(f"  A={a:3d}, R={r:.2f}")

Found 0 completed runs out of 110 total.
Remaining to run: 110


## Cell 5 — Build Run Queue & Estimate Runtime

In [7]:
# ========================================================================
# BUILD ORDERED RUN QUEUE
# ========================================================================

run_queue = []
for n_agents in AGENT_COUNTS:
    for rr in RECESSION_RATES:
        key = (n_agents, round(rr, 2))
        if SKIP_COMPLETED and key in completed_runs:
            continue
        run_queue.append({
            'n_agents': n_agents,
            'recession_rate': rr,
        })

print(f"Run queue: {len(run_queue)} runs")
print(f"Estimated time: {len(run_queue) * 8:.0f} minutes ({len(run_queue) * 8 / 60:.1f} hours) @ ~8 min/run avg")

# Preview first 10
print("\nFirst 10 in queue:")
for i, r in enumerate(run_queue[:10]):
    print(f"  {i+1:3d}. A={r['n_agents']:3d}, R={r['recession_rate']:.2f}")
if len(run_queue) > 10:
    print(f"  ... and {len(run_queue) - 10} more")

Run queue: 110 runs
Estimated time: 880 minutes (14.7 hours) @ ~8 min/run avg

First 10 in queue:
    1. A=  2, R=0.10
    2. A=  2, R=0.20
    3. A=  2, R=0.30
    4. A=  2, R=0.40
    5. A=  2, R=0.50
    6. A=  2, R=0.60
    7. A=  2, R=0.70
    8. A=  2, R=0.80
    9. A=  2, R=0.90
   10. A=  2, R=1.00
  ... and 100 more


## Cell 6 — Load Input Data

In [8]:
# ========================================================================
# LOAD INPUT CSV (once — shared across all runs)
# ========================================================================

df_world = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Loaded {len(df_world)} rows from {CSV_PATH}")
print(f"Columns: {list(df_world.columns)}")

# Pre-validate
required = ['x', 'y', 'elevation', 'drainage_area_m2', 'stream']
missing = [c for c in required if c not in df_world.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

Loaded 1007984 rows from C:\Users\Wyss User\Desktop\DigitalBeaverWorld\A_MASTER_veg_stream.csv
Columns: ['x', 'y', 'elevation', 'drainage_area_m2', 'slope', 'width_m', 'depth_m', 'percentile', 'streamline', 'label', 'slope_degrees', 'traversable', 'naip_R', 'naip_G', 'naip_B', 'naip_NIR', 'lon', 'lat', 'veg_NDVI', 'veg_VARI', 'veg_NGRDI', 'veg_index_primary', 'veg_quality_01', 'veg_class', 'stream']


## Cell 7 — BATCH EXECUTION

**This is the main loop.** Each run:
1. Builds model kwargs for the current (n_agents, recession_rate) combo
2. Calls `run_yearly_cycles_v2()` for N_YEARS
3. Logs timing and progress
4. Continues to next run

If the kernel crashes or you interrupt, just re-run — completed runs are skipped.

In [9]:
# ========================================================================
# BATCH EXECUTION LOOP
# ========================================================================

batch_start = time.time()
batch_log = []  # (n_agents, recession_rate, elapsed_sec, status)

# Ensure global for model plotting
PLOT_END_OF_YEAR = PLOT_DURING_RUN

print(f"{'='*80}")
print(f"  BATCH START: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Runs to execute: {len(run_queue)}")
print(f"{'='*80}\n")

for run_idx, run_params in enumerate(run_queue):
    n_agents = run_params['n_agents']
    rr = run_params['recession_rate']
    run_start = time.time()

    print(f"\n{'━'*80}")
    print(f"  RUN {run_idx + 1}/{len(run_queue)}: "
          f"A={n_agents}, R={rr:.2f} | "
          f"{datetime.datetime.now().strftime('%H:%M:%S')}")
    print(f"{'━'*80}")

    # Double-check not already completed (in case of parallel runs)
    if SKIP_COMPLETED:
        current_completed = find_completed_runs(RUNS_BASE_DIR, N_YEARS)
        if (n_agents, round(rr, 2)) in current_completed:
            print(f"  → Already completed, skipping.")
            batch_log.append((n_agents, rr, 0, 'skipped'))
            continue

    try:
        model_kwargs = dict(
            n_agents=n_agents,
            ticks_per_year=TICKS_PER_YEAR,
            excavation_depth_m=EXCAVATION_DEPTH_M,
            agent_speed=AGENT_SPEED,
            recession_rate=rr,
            vision_min=VISION_MIN,
            vision_max=VISION_MAX,
            verbose=VERBOSE_MODEL,
        )

        outputs, df_final = run_yearly_cycles_v2(
            df0=df_world.copy(),
            original_csv_path=CSV_PATH,
            runs_base_dir=RUNS_BASE_DIR,
            n_years=N_YEARS,
            model_kwargs=model_kwargs,
            data_epsg=DATA_EPSG,
            grid_spacing_m=GRID_SPACING_M,
            width_depth_fn=None,
            r_limit=R_LIMIT,
            save_drainage_fig=False,     # Skip plots for speed
            save_dem_each_year=False,     # Skip DEM PNGs for speed
            save_excavation_timeseries=True,
        )

        elapsed = time.time() - run_start
        batch_log.append((n_agents, rr, elapsed, 'success'))
        print(f"\n  ✓ COMPLETED in {elapsed/60:.1f} min")

    except Exception as e:
        elapsed = time.time() - run_start
        batch_log.append((n_agents, rr, elapsed, f'FAILED: {e}'))
        print(f"\n  ✗ FAILED after {elapsed/60:.1f} min: {e}")
        import traceback
        traceback.print_exc()
        # Continue to next run — don't crash the whole batch
        continue

    # Progress estimate
    total_elapsed = time.time() - batch_start
    runs_done = run_idx + 1
    runs_left = len(run_queue) - runs_done
    avg_per_run = total_elapsed / runs_done
    eta_sec = avg_per_run * runs_left
    eta_str = str(datetime.timedelta(seconds=int(eta_sec)))
    print(f"  Progress: {runs_done}/{len(run_queue)} "
          f"({runs_done/len(run_queue)*100:.0f}%) | "
          f"Avg: {avg_per_run/60:.1f} min/run | "
          f"ETA: {eta_str}")

# ========================================================================
# BATCH SUMMARY
# ========================================================================
total_time = time.time() - batch_start
print(f"\n{'='*80}")
print(f"  BATCH COMPLETE: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"  Total time: {total_time/3600:.1f} hours ({total_time/60:.0f} minutes)")
print(f"{'='*80}")

# Tally results
n_success = sum(1 for _, _, _, s in batch_log if s == 'success')
n_skipped = sum(1 for _, _, _, s in batch_log if s == 'skipped')
n_failed = sum(1 for _, _, _, s in batch_log if s.startswith('FAILED'))
print(f"  Successful: {n_success}")
print(f"  Skipped:    {n_skipped}")
print(f"  Failed:     {n_failed}")

if n_failed > 0:
    print("\n  Failed runs:")
    for a, r, t, s in batch_log:
        if s.startswith('FAILED'):
            print(f"    A={a:3d}, R={r:.2f}: {s}")

  BATCH START: 2026-02-25 21:06:17
  Runs to execute: 110


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  RUN 1/110: A=2, R=0.10 | 21:06:17
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[RUN OUTPUT] H:\DigitalBeaverWorld\runs\A_MASTER_veg_stream_Y10_A2_R0p1
  Original DEM cached: (1168, 863), dx=1.0, dy=1.0
  r_limit = 6 m (hemisphere rescaling ceiling)

  YEAR 1/10 — Hemisphere Erosion Pipeline
  site_counter_dem: A_MASTER_veg_stream_site_counter_dem_year01.csv
    Total cumulative excavation: 437.80
  site_volume: A_MASTER_veg_stream_site_volume_year01.csv
    Active cells: 2189, r_limit achieved: 0.346 m
  2d_erosion: A_MASTER_veg_stream_2d_erosion_year01.csv
    Max hemisphere carving depth: 0.346 m
    Erosion -> hydro mapping: 1007984/1007984 cells matched
  Updated (hydrology on 2d_erosion): A_MASTER_veg_stream_year1.csv
Saved H:\DigitalBeaverWorld\runs\A_MASTER_veg_stream_Y10_A2_R0p1\Excavation heatmaps\A_

## Cell 8 — Save Batch Log & Timing

In [10]:
# ========================================================================
# SAVE BATCH LOG
# ========================================================================

df_log = pd.DataFrame(batch_log, columns=['n_agents', 'recession_rate', 'elapsed_sec', 'status'])
df_log['elapsed_min'] = df_log['elapsed_sec'] / 60.0

log_path = Path(RUNS_BASE_DIR) / 'batch_log.csv'
df_log.to_csv(log_path, index=False)
print(f"Saved batch log: {log_path}")

# Timing heatmap
successful = df_log[df_log['status'] == 'success'].copy()
if len(successful) > 2:
    fig, ax = plt.subplots(figsize=(12, 6))

    agents_u = sorted(successful['n_agents'].unique())
    rr_u = sorted(successful['recession_rate'].unique())
    timing_grid = np.full((len(agents_u), len(rr_u)), np.nan)

    for i, a in enumerate(agents_u):
        for j, r in enumerate(rr_u):
            mask = (successful['n_agents'] == a) & (np.abs(successful['recession_rate'] - r) < 0.01)
            if mask.any():
                timing_grid[i, j] = successful.loc[mask, 'elapsed_min'].values[0]

    im = ax.imshow(timing_grid, cmap='YlOrRd', aspect='auto', origin='lower')
    ax.set_xticks(range(len(rr_u)))
    ax.set_xticklabels([f'{r:.2f}' for r in rr_u], rotation=45, ha='right')
    ax.set_yticks(range(len(agents_u)))
    ax.set_yticklabels(agents_u)
    ax.set_xlabel('Recession Rate')
    ax.set_ylabel('Number of Agents')
    ax.set_title('Run Time (minutes) per Parameter Combination')
    plt.colorbar(im, ax=ax, label='Minutes')
    plt.tight_layout()
    plt.savefig(Path(RUNS_BASE_DIR) / 'batch_timing_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nTiming stats:")
    print(f"  Mean:   {successful['elapsed_min'].mean():.1f} min")
    print(f"  Median: {successful['elapsed_min'].median():.1f} min")
    print(f"  Min:    {successful['elapsed_min'].min():.1f} min")
    print(f"  Max:    {successful['elapsed_min'].max():.1f} min")
else:
    print("Not enough successful runs for timing analysis.")

df_log

Saved batch log: H:\DigitalBeaverWorld\runs\batch_log.csv

Timing stats:
  Mean:   28.0 min
  Median: 26.1 min
  Min:    20.9 min
  Max:    52.0 min


C:\Users\Wyss User\AppData\Local\Temp\ipykernel_55484\2608393864.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,n_agents,recession_rate,elapsed_sec,status,elapsed_min
0,2,0.1,1324.540364,success,22.075673
1,2,0.2,1385.640152,success,23.094003
2,2,0.3,1412.014692,success,23.533578
3,2,0.4,1307.209536,success,21.786826
4,2,0.5,1257.546329,success,20.959105
...,...,...,...,...,...
105,25,0.6,2278.490335,success,37.974839
106,25,0.7,2590.917043,success,43.181951
107,25,0.8,2776.513105,success,46.275218
108,25,0.9,2966.643402,success,49.444057


## Cell 9 — Verify All Runs Completed

In [11]:
# ========================================================================
# VERIFY COMPLETENESS OF PARAMETER SPACE
# ========================================================================

final_completed = find_completed_runs(RUNS_BASE_DIR, N_YEARS)

print(f"Completed: {len(final_completed)}/{total_runs}")

# Check for gaps
missing = []
for a in AGENT_COUNTS:
    for rr in RECESSION_RATES:
        key = (a, round(rr, 2))
        if key not in final_completed:
            missing.append(key)

if missing:
    print(f"\n⚠ Missing {len(missing)} runs:")
    for a, r in missing:
        print(f"  A={a:3d}, R={r:.2f}")
    print("\nRe-run this notebook to fill gaps (completed runs will be skipped).")
else:
    print("\n✓ All 110 runs complete! Ready for analysis.")
    print(f"  → Open Canal_Formation_Analysis.ipynb to generate figures.")

# Visual completeness grid
fig, ax = plt.subplots(figsize=(12, 6))
grid = np.zeros((len(AGENT_COUNTS), len(RECESSION_RATES)))
for i, a in enumerate(AGENT_COUNTS):
    for j, rr in enumerate(RECESSION_RATES):
        if (a, round(rr, 2)) in final_completed:
            grid[i, j] = 1

cmap = plt.cm.colors.ListedColormap(['#FFCDD2', '#C8E6C9'])  # red=missing, green=done
ax.imshow(grid, cmap=cmap, aspect='auto', origin='lower', vmin=0, vmax=1)
ax.set_xticks(range(len(RECESSION_RATES)))
ax.set_xticklabels([f'{r:.2f}' for r in RECESSION_RATES], rotation=45, ha='right')
ax.set_yticks(range(len(AGENT_COUNTS)))
ax.set_yticklabels(AGENT_COUNTS)
ax.set_xlabel('Recession Rate (slow/wet → fast/drought)', fontsize=11)
ax.set_ylabel('Number of Agents', fontsize=11)
ax.set_title(f'Run Completeness: {len(final_completed)}/{total_runs} '
             f'({len(final_completed)/total_runs*100:.0f}%)',
             fontsize=13, fontweight='bold')

# Annotate cells
for i in range(len(AGENT_COUNTS)):
    for j in range(len(RECESSION_RATES)):
        symbol = '✓' if grid[i, j] == 1 else '✗'
        color = 'darkgreen' if grid[i, j] == 1 else 'red'
        ax.text(j, i, symbol, ha='center', va='center', fontsize=12,
                fontweight='bold', color=color)

plt.tight_layout()
plt.savefig(Path(RUNS_BASE_DIR) / 'batch_completeness.png', dpi=150, bbox_inches='tight')
plt.show()

Completed: 110/110

✓ All 110 runs complete! Ready for analysis.
  → Open Canal_Formation_Analysis.ipynb to generate figures.


C:\Users\Wyss User\AppData\Local\Temp\ipykernel_55484\3179946045.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
